# Chain-of-Thought Faithfulness — CS60075 Assignment 2, Project 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chimankarparag/cs60075-cot-faithfulness/blob/main/code/CoT_Faithfulness_Colab.ipynb)

**Parag Chimankar (23CS10049)** · IIT Kharagpur · CS60075 NLP

End-to-end notebook for all three tasks on the assigned dataset, using `meta-llama/Llama-3.1-8B-Instruct`:

1. **Task 1** — direct (no-CoT) answering baseline, zero-shot and 4-shot
2. **Task 2** — CoT faithfulness interventions (Lanham et al., 2023): early answering, adding a mistake, paraphrasing, filler tokens
3. **Task 3** — faithfulness-weighted self-consistency

> **Status:** skeleton only. Code cells are added once the modules in `code/` exist.

## How to run this notebook

**In VS Code (Colab extension):** open this file → *Select Kernel* → *Colab* → sign in with Google → choose a GPU runtime.
**In the browser:** open it from GitHub in Colab → *Runtime → Change runtime type → GPU*.

The Colab runtime does **not** see files on your laptop. The setup cell clones the GitHub repo and downloads the data itself.

| Setting | Meaning |
| --- | --- |
| `DATASET` | `gsm8k`, `bbh` or `strategyqa` — the group's assigned dataset |
| `REUSE_RESULTS` | `True` = rebuild all tables from the committed generations in `results/` (minutes, no big GPU needed); `False` = regenerate |
| `N_LIMIT` | Optional small n for a quick smoke test |

GPU note: a free T4 (~15 GB) cannot hold the 8B model in 16-bit, so on a T4 the model is loaded in 4-bit NF4. On L4/A100 it runs in bf16. Headline numbers in the report come from bf16 runs on the IIT KGP GPU servers.

## 0. Setup

Planned cells:

1. Check the GPU (`nvidia-smi`) and pick precision.
2. Hugging Face login with the `HF_TOKEN` secret (Llama-3.1-8B-Instruct is gated).
3. Clone the repo (GitHub token needed while it is private) and `pip install -r requirements.txt`.
4. Download the three datasets (pinned revisions, checksums in `data/MANIFEST.md`) and build the eval / few-shot files with seed 42.

## 1. Task 1 — Direct answering baseline (30 marks)

- Greedy decoding, `max_new_tokens` = 16, answer-only instruction.
- Zero-shot and 4-shot (same 4 exemplars for every question).
- **Reports:** accuracy (zero-shot, 4-shot) with 95% CI; parse-failure rate.

## 2. Task 2 — CoT faithfulness interventions (30 marks)

### 2.1 Standard CoT
"Let's think step by step", temperature 0.7, top-p 0.95, max 512 new tokens. **Reports:** CoT accuracy and gain over Task 1's best variant.

### 2.2 Early answering
Truncate at 20 / 40 / 60 / 80 % of sentences and force an answer (0 % and 100 % also run for the curve).

### 2.3 Adding a mistake
Corrupt one reasoning step, regenerate the remainder, re-answer.

### 2.4 Paraphrasing
Reword the reasoning without changing its content, re-answer.

### 2.5 Filler tokens
Replace the reasoning with `" ..."` repeated to the same token length, re-answer.

### 2.6 Faithfulness scores and comparison with the paper
Score per intervention = share of items whose answer changes (or, for paraphrasing, stays the same) as it should. Early-answering AOC compared with Lanham et al. Table 2.

## 3. Task 3 — Faithfulness-weighted self-consistency (40 marks)

1. Sample k = 5 CoTs per question (temperature 0.7).
2. Early-answer each sample at 50 % truncation.
3. Weighted majority vote (spec rule as main result, Lanham-consistent rule as ablation; plain self-consistency as a control).
4. Accuracy vs Tasks 1 and 2.
5. All four interventions on the pipeline's chosen CoT → aggregate faithfulness vs Task 2.

## 4. Summary

Final tables and plots for the report: Task 1 vs Task 2 vs Task 3 accuracy, per-intervention faithfulness, early-answering curve, and α sensitivity.